# 对分易自动签到 APK 构建
全部运行 → 授权 Drive → 等 20-30 分钟 → APK 存到网盘。关电脑不受影响。

**如果构建失败**: 检查最后输出的错误摘要，常见问题是 Cython 版本不兼容（已修复）。

In [ ]:
# 0. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. 安装系统依赖
!sudo apt-get update -qq
!sudo apt-get install -y -qq \
    openjdk-17-jdk-headless \
    autoconf automake libtool \
    cmake zip unzip pkg-config \
    libffi-dev libssl-dev \
    libxml2-dev libxslt1-dev \
    python3-pip git \
    libmtdev-dev libx11-dev libxi-dev \
    libgl1-mesa-dev libgles2-mesa-dev \
    libsdl2-dev libsdl2-image-dev libsdl2-mixer-dev libsdl2-ttf-dev

In [ ]:
# 2. 安装 buildozer + cython（必须锁定版本）
!python3 -m pip install -q buildozer "cython==0.29.36" legacy-cgi
!/usr/bin/python3 -m pip install -q buildozer "cython==0.29.36" legacy-cgi

# 给 python-for-android 的宿主 Python 和它创建的虚拟环境补 cgi.py，兼容 Python 3.13+
import sys, subprocess, pathlib, textwrap

cgi_source = textwrap.dedent('''
    import html
    import urllib.parse

    def escape(s, quote=True):
        return html.escape(s, quote=quote)

    def parse_qs(qs, keep_blank_values=False, strict_parsing=False):
        return urllib.parse.parse_qs(qs, keep_blank_values=keep_blank_values, strict_parsing=strict_parsing)

    def parse_qsl(qs, keep_blank_values=False, strict_parsing=False):
        return urllib.parse.parse_qsl(qs, keep_blank_values=keep_blank_values, strict_parsing=strict_parsing)

    def parse_header(line):
        parts = [p.strip() for p in line.split(';')]
        key = parts[0].lower()
        params = {}
        for part in parts[1:]:
            if '=' in part:
                name, value = part.split('=', 1)
                params[name.strip().lower()] = value.strip().strip('"')
        return key, params
''').strip() + '\n'

for key in ('stdlib', 'purelib'):
    target_dir = subprocess.check_output(
        ['/usr/bin/python3', '-c', f'import sysconfig; print(sysconfig.get_paths()["{key}"])'],
        text=True,
    ).strip()
    cgi_path = pathlib.Path(target_dir) / 'cgi.py'
    if not cgi_path.exists():
        cgi_path.write_text(cgi_source)
    print(f'{key} cgi: {cgi_path}')

# 检查版本和 Python 3.13+ 兼容补丁
import cython, buildozer, cgi
print(f"Notebook Python: {sys.executable}")
print(f"Python: {sys.version}")
print(f"Cython: {cython.__version__}")
print(f"Buildozer: {buildozer.__version__}")
print(f"cgi module: {cgi.__file__}")
subprocess.run(['/usr/bin/python3', '-c', 'import sys, cgi; print("Buildozer Python:", sys.executable); print("Buildozer cgi:", cgi.__file__)'], check=True)

In [ ]:
# 3. 克隆项目
!rm -rf duifene-sign
!git clone https://github.com/Mafeixxn/duifene-sign.git
%cd duifene-sign/android

In [ ]:
# 4. 构建 APK（约 15-30 分钟）
import subprocess, os, shutil

os.environ['BUILDOZER_ALLOW_ROOT'] = '1'

log_file = 'build_1.log'
print(f'>>> 开始构建（日志: {log_file}）...')
print(f'磁盘剩余: {shutil.disk_usage(".").free // (1024**3)} GB')

with open(log_file, 'w') as f:
    r = subprocess.run(
        ['buildozer', '--verbose', 'android', 'debug'],
        env={**os.environ, 'BUILDOZER_ALLOW_ROOT': '1'},
        input=b'y\n', stdout=f, stderr=subprocess.STDOUT,
    )

if r.returncode == 0:
    print(f'✅ 构建成功！日志: {log_file}')
else:
    print(f'❌ 构建失败 (exit={r.returncode})，请运行下一个单元查看错误')

In [ ]:
# 5. 显示最近的错误（帮助排查）
import glob
logs = sorted(glob.glob('build_*.log'))
if logs:
    latest = logs[-1]
    print(f'--- {latest} 关键错误 ---')
    !grep -i -E 'error:|failed:|exception|traceback|command failed|No module named|No such file|Could not|not found|BUILD FAILED|FAILURE:' {latest} | tail -80 || true
    print(f'\n--- {latest} 最后 80 行 ---')
    !tail -80 {latest}
else:
    print('❌ 未找到 build_*.log，请先运行构建单元')

In [ ]:
# 6. 保存 APK 到 Google Drive
import glob, shutil, os

apks = glob.glob('bin/*.apk')
if apks:
    dest = '/content/drive/MyDrive/duifene_sign.apk'
    shutil.copy(apks[0], dest)
    mb = os.path.getsize(apks[0]) / (1024*1024)
    print(f'✅ APK: {apks[0]} ({mb:.1f}MB) → Drive 根目录 duifene_sign.apk')
else:
    print('❌ 未找到 APK，构建可能失败')
    print('请检查上面的 build_*.log，或 Drive 中的 duifene_build_*.log')